# MDD Challenge 2025 — Training + Public Test Evaluation

Full pipeline:
1. Fine-tune `Wav2Vec2ForCTC` trên tập train (từ splits/)
2. Mỗi epoch: đánh giá trên validation, lưu best checkpoint
3. Sau training: tính FP rates trên validation, chạy K=50 calibration
4. Inference trên public test → score → ghi `result_public.csv`

**Kaggle datasets cần add:**
- `honggiangtrnh/mdd-challenge-2025` — training + public test data
- *(optional)* `trngquangthi4139/mdd-public-test` — nếu public test nằm tách riêng

In [ ]:
# Cài thêm nếu cần (Kaggle đã có sẵn transformers, torch)
# !pip install -q transformers==4.40.0 accelerate
import os, sys, math, wave, json, csv, re, warnings
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd
import torch
import scipy.special
from torch.utils.data import Dataset, RandomSampler, SequentialSampler
from transformers import (
    Wav2Vec2ForCTC, AutoFeatureExtractor,
    TrainerCallback, TrainingArguments, Trainer,
    get_linear_schedule_with_warmup,
)
try:
    from transformers.trainer_utils import LengthGroupedSampler
except ImportError:
    from transformers.trainer_pt_utils import LengthGroupedSampler

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
BASE_MODEL   = 'nguyenvulebinh/wav2vec2-base-vietnamese-250h'
EXP_NAME     = 'run01'
EPOCHS       = 10
BATCH_SIZE   = 4
GRAD_ACCUM   = 4          # effective batch = 16
ENCODER_LR   = 2e-5
HEAD_LR      = 2e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
SEED         = 42
ENABLE_SPEC_AUGMENT = False

# Calibration
K_SUPPRESS  = 50
DEF_THR     = 0.90
MIN_FP      = 5
INFER_BATCH = 16

BLANK = '<blank>'
UNK   = '<unk>'
VALID_SPEAKERS = frozenset(['S0008', 'S0003'])

In [ ]:
# ── Paths (Kaggle + local fallback) ─────────────────────────────────────────
def _find(*candidates):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    raise FileNotFoundError(f'None of these paths exist: {candidates}')

DATA_ROOT = _find(
    '/kaggle/input/mdd-challenge-2025',
    '.',
)
TRAIN_ROOT = _find(
    DATA_ROOT / 'MDD-Challenge-2025-training-set',
    'MDD-Challenge-2025-training-set',
)
AUDIO_DIR = TRAIN_ROOT / 'audio_data' / 'train'
META_DIR  = TRAIN_ROOT / 'metadata'

PUBTEST_ROOT = _find(
    '/kaggle/input/mdd-public-test/MDD-Challenge-2025-public-test',
    DATA_ROOT / 'MDD-Challenge-2025-public-test',
    'MDD-Challenge-2025-public-test',
)
PUBTEST_AUDIO = PUBTEST_ROOT / 'audio_data' / 'public_test'
PUBTEST_META  = PUBTEST_ROOT / 'metadata'

OUT_DIR      = Path('/kaggle/working') / EXP_NAME
BEST_CKP_DIR = OUT_DIR / 'best_ckp'
OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = Path('/kaggle/working/result_public.csv')

print(f'Audio dir   : {AUDIO_DIR}  exists={AUDIO_DIR.exists()}')
print(f'PubTest audio: {PUBTEST_AUDIO}  exists={PUBTEST_AUDIO.exists()}')
print(f'Output dir  : {OUT_DIR}')

In [ ]:
# ── Audio utilities ──────────────────────────────────────────────────────────
def load_wav_f32(path):
    """Load WAV -> float32 [-1, 1], returns (array, sample_rate)."""
    with wave.open(str(path), 'rb') as wf:
        sr, sw, nch = wf.getframerate(), wf.getsampwidth(), wf.getnchannels()
        frames = wf.readframes(wf.getnframes())
    dtype = {1: np.uint8, 2: np.int16, 4: np.int32}[sw]
    scale = {1: 128.0,   2: 32768.0,  4: 2147483648.0}[sw]
    y = np.frombuffer(frames, dtype=dtype).astype(np.float32)
    y = (y - 128.0) / 128.0 if sw == 1 else y / scale
    if nch > 1:
        y = y.reshape(-1, nch).mean(axis=1)
    return y.copy(), sr

def normalize_amp(y, target_peak=0.9):
    peak = np.abs(y).max()
    return y if peak < 1e-6 else y * (target_peak / peak)

def trim_silence(y, sr=16000, threshold_db=-45.0,
                 frame_ms=25, hop_ms=10, min_dur_sec=0.3):
    fl = int(sr * frame_ms / 1000)
    hl = int(sr * hop_ms  / 1000)
    energies = [np.mean(y[i:i+fl]**2) for i in range(0, max(1, len(y)-fl), hl)]
    db  = 10 * np.log10(np.array(energies) + 1e-10)
    act = db > threshold_db
    if not act.any():
        return y
    s = max(0, int(np.argmax(act)) * hl - fl)
    e = min(len(y), (len(act) - int(np.argmax(act[::-1]))) * hl + fl)
    trimmed = y[s:e]
    return y if len(trimmed) / sr < min_dur_sec else trimmed

def aug_gain(y, lo=-6., hi=6.):
    return np.clip(y * 10 ** (np.random.uniform(lo, hi) / 20.), -1., 1.)

def aug_noise(y, snr_lo=20., snr_hi=35., prob=0.3):
    if np.random.rand() > prob:
        return y
    sp  = np.mean(y ** 2) + 1e-10
    np_ = sp / 10 ** (np.random.uniform(snr_lo, snr_hi) / 10.)
    return np.clip(y + np.random.normal(0., np_**0.5, y.shape).astype(np.float32), -1., 1.)

def norm_phones(s):
    return ' '.join(str(s).replace('*', '').replace('$', '').split())

def tokenize(s):
    return norm_phones(s).split()

def get_speaker_id(path):
    stem = Path(path).stem
    m = re.search(r'(S\d+)', stem)
    if m: return m.group(1)
    return 'TUYEN' if 'tuyen' in stem.lower() else 'ADULT'

def preproc(y):
    return normalize_amp(trim_silence(y))

def augment(y):
    return aug_noise(aug_gain(y))

print('Audio utilities ready.')

In [ ]:
# ── Evaluate functions (inline từ evaluate.py) ───────────────────────────────
def _align(seq1, seq2):
    GAP = -1; MATCH = 1; MISMATCH = -1
    n, m = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): score[i][0] = GAP * i
    for j in range(n + 1): score[0][j] = GAP * j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            s = MATCH if seq1[j-1] == seq2[i-1] else \
                GAP if (seq1[j-1] == '<eps>' or seq2[i-1] == '<eps>') else MISMATCH
            score[i][j] = max(score[i-1][j-1]+s, score[i-1][j]+GAP, score[i][j-1]+GAP)
    align1, align2 = [], []
    i, j = m, n
    while i > 0 and j > 0:
        s = MATCH if seq1[j-1] == seq2[i-1] else \
            GAP if (seq1[j-1] == '<eps>' or seq2[i-1] == '<eps>') else MISMATCH
        if score[i][j] == score[i-1][j-1] + s:
            align1.append(seq1[j-1]); align2.append(seq2[i-1]); i -= 1; j -= 1
        elif score[i][j] == score[i][j-1] + GAP:
            align1.append(seq1[j-1]); align2.append('<eps>'); j -= 1
        else:
            align1.append('<eps>'); align2.append(seq2[i-1]); i -= 1
    while j > 0: align1.append(seq1[j-1]); align2.append('<eps>'); j -= 1
    while i > 0: align1.append('<eps>'); align2.append(seq2[i-1]); i -= 1
    align1.reverse(); align2.reverse()
    return align1, align2

def _ops(a1, a2):
    ops = []
    for r, h in zip(a1, a2):
        if   r != '<eps>' and h == '<eps>': ops.append('D')
        elif r == '<eps>' and h != '<eps>': ops.append('I')
        elif r != h:                        ops.append('S')
        else:                               ops.append('C')
    return ops

def _align_pair(s1, s2):
    seq1 = s1.replace('*','').replace('$','').split()
    seq2 = s2.replace('*','').replace('$','').split()
    a1, a2 = _align(seq1, seq2)
    return a1, a2, _ops(a1, a2)

def _score_preds(gt_c, gt_t, preds):
    cor_cor=cor_nocor=0
    sub_sub=sub_sub1=sub_nosub=0
    ins_ins=ins_ins1=ins_noins=0
    del_del=del_del1=del_nodel=0
    total_sub=total_del=total_ins=total_cor_per=0
    sub_sub_de=del_del_de=ins_ins_de=0

    for c, t, p in zip(gt_c, gt_t, preds):
        rs, hs, op_rh   = _align_pair(c, t)
        hs2, os2, op_ho = _align_pair(t, p)
        rs3, os3, op_ro = _align_pair(c, p)

        # PER: align transcript vs prediction
        total_sub += op_ho.count('S')
        total_del += op_ho.count('D')
        total_ins += op_ho.count('I')
        total_cor_per += op_ho.count('C')

        # Deletion detection
        flag = 0
        for i in range(len(rs)):
            if rs[i] == '<eps>': continue
            while flag < len(rs3) and rs3[flag] == '<eps>': flag += 1
            if flag < len(rs3) and rs[i] == rs3[flag]:
                if   op_rh[i]=='D' and op_ro[flag]=='D':              del_del  += 1
                elif op_rh[i]=='D' and op_ro[flag] not in ('D','C'): del_del1 += 1
                elif op_rh[i]=='D' and op_ro[flag]=='C':              del_nodel+= 1
                flag += 1

        # Correct / Sub / Ins detection
        flag = 0
        for i in range(len(hs)):
            if hs[i] == '<eps>': continue
            while flag < len(hs2) and hs2[flag] == '<eps>': flag += 1
            if flag < len(hs2) and hs[i] == hs2[flag]:
                if   op_rh[i]=='C' and op_ho[flag]=='C':  cor_cor  += 1
                elif op_rh[i]=='C' and op_ho[flag]!='C':  cor_nocor+= 1
                if   op_rh[i]=='S' and op_ho[flag]=='C':  sub_sub  += 1
                elif op_rh[i]=='S' and op_ho[flag]!='C' and rs[i]!=os2[flag]: sub_sub1 += 1
                elif op_rh[i]=='S' and op_ho[flag]!='C' and rs[i]==os2[flag]: sub_nosub+= 1
                if   op_rh[i]=='I' and op_ho[flag]=='C':  ins_ins  += 1
                elif op_rh[i]=='I' and op_ho[flag]!='C' and op_ho[flag]!='D': ins_ins1 += 1
                elif op_rh[i]=='I' and op_ho[flag]=='D':  ins_noins+= 1
                flag += 1

    TR = sub_sub+sub_sub1+del_del+del_del1+ins_ins+ins_ins1
    FR = cor_nocor
    FA = sub_nosub+ins_noins+del_nodel
    DE = sub_sub1+del_del1+ins_ins1
    ref_len = total_sub+total_del+total_cor_per

    prec = TR/(TR+FR) if (TR+FR)>0 else 0.
    rec  = TR/(TR+FA) if (TR+FA)>0 else 0.
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.
    per  = (total_sub+total_del+total_ins)/ref_len if ref_len>0 else 0.
    der  = DE/(TR+FA) if (TR+FA)>0 else 0.
    sc   = 0.5*f1 + 0.4*(1-der) + 0.1*(1-per)
    return {'f1':f1,'prec':prec,'rec':rec,'per':per,'der':der,'score':sc}

def evaluate_on_valid(gt_c, gt_t, preds, tag=''):
    m = _score_preds(gt_c, gt_t, preds)
    if tag:
        print(f'{tag:50s}  F1={m["f1"]:.4f}  PER={m["per"]:.4f}  '
              f'DER={m["der"]:.4f}  Score={m["score"]:.4f}')
    return m

print('Evaluate functions ready.')

In [ ]:
# ── Load data + build vocab + split ─────────────────────────────────────────
df_t = pd.read_csv(META_DIR / 'train.csv')
df_p = pd.read_csv(META_DIR / 'train_phones.csv')
print(f'Loaded {len(df_t)} total samples')

df_t['speaker_id'] = df_t['path'].map(get_speaker_id)
df_p['speaker_id'] = df_t['speaker_id'].values
df_p['c_norm']     = df_p['canonical'].map(norm_phones)
df_p['t_norm']     = df_p['transcript'].map(norm_phones)
df_p['ph_error']   = (df_p['c_norm'] != df_p['t_norm'])

valid_mask = df_t['speaker_id'].isin(VALID_SPEAKERS)
train_df = df_t[~valid_mask].reset_index(drop=True)
valid_df = df_t[ valid_mask].reset_index(drop=True)
train_ph = df_p[~valid_mask].reset_index(drop=True)
valid_ph = df_p[ valid_mask].reset_index(drop=True)

assert set(train_df['speaker_id']).isdisjoint(VALID_SPEAKERS), 'Speaker leak!'
print(f'Train: {len(train_df)}  Valid: {len(valid_df)}')
print(f'Train error rate: {train_ph["ph_error"].mean():.4f}')
print(f'Valid error rate: {valid_ph["ph_error"].mean():.4f}')

# Build vocab (transcript from train only + canonical from all)
counter = Counter()
for s in train_ph['t_norm']:  counter.update(tokenize(s))
for s in df_p['c_norm']:      counter.update(tokenize(s))
id2phone = [BLANK, UNK] + sorted(counter.keys())
phone2id = {p: i for i, p in enumerate(id2phone)}
BLANK_ID = phone2id[BLANK]

print(f'Vocab size: {len(id2phone)}')

VALID_C = valid_ph['c_norm'].tolist()
VALID_T = valid_ph['t_norm'].tolist()
val_paths = [AUDIO_DIR / Path(p).name for p in valid_ph['path']]

In [ ]:
# ── Dataset + Collator + MDDTrainer ─────────────────────────────────────────
class MDDDataset(Dataset):
    def __init__(self, df, ph_df, is_train=True):
        self.df       = df.reset_index(drop=True)
        self.ph       = ph_df.reset_index(drop=True)
        self.is_train = is_train
        # Precompute lengths for group_by_length (reads WAV headers, no decoding)
        self.lengths = []
        for _, row in self.df.iterrows():
            p = AUDIO_DIR / Path(row['path']).name
            try:
                with wave.open(str(p), 'rb') as wf:
                    self.lengths.append(wf.getnframes())
            except Exception:
                self.lengths.append(0)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ph  = self.ph.iloc[idx]
        y, _ = load_wav_f32(AUDIO_DIR / Path(row['path']).name)
        y = preproc(y)
        if self.is_train:
            y = augment(y)
        labels = [phone2id.get(t, phone2id[UNK]) for t in tokenize(ph['t_norm'])]
        return {'input_values': y.astype(np.float32), 'labels': labels}


@dataclass
class MDDCollator:
    fe: object
    sr: int = 16000
    pad_id: int = -100

    def __call__(self, batch):
        out = self.fe(
            [b['input_values'] for b in batch],
            sampling_rate=self.sr, padding=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        labels = [torch.tensor(b['labels'], dtype=torch.long) for b in batch]
        maxL = max(len(l) for l in labels)
        pad  = torch.full((len(labels), maxL), self.pad_id, dtype=torch.long)
        for i, l in enumerate(labels): pad[i, :len(l)] = l
        out['labels'] = pad
        return out


class MDDTrainer(Trainer):
    """Fix group_by_length cho plain torch Dataset (Trainer chỉ support HF Dataset)."""
    def _get_train_sampler(self, train_dataset=None):
        ds = train_dataset or self.train_dataset
        if not self.args.group_by_length or not hasattr(ds, 'lengths'):
            return RandomSampler(ds)
        return LengthGroupedSampler(
            self.args.train_batch_size * self.args.gradient_accumulation_steps,
            lengths=ds.lengths,
            dataset=ds,
        )
    def _get_eval_sampler(self, eval_dataset=None):
        return SequentialSampler(eval_dataset or self.eval_dataset)

print('Dataset / Collator / Trainer ready.')

In [ ]:
# ── Build model + optimizer + scheduler ─────────────────────────────────────
def build_model():
    fe = AutoFeatureExtractor.from_pretrained(BASE_MODEL)
    model = Wav2Vec2ForCTC.from_pretrained(
        BASE_MODEL,
        vocab_size=len(id2phone),
        pad_token_id=BLANK_ID,
        ctc_loss_reduction='mean',
        ctc_zero_infinity=True,
        ignore_mismatched_sizes=True,
    )
    model.config.apply_spec_augment = ENABLE_SPEC_AUGMENT
    model.config.mask_time_prob     = 0.05 if ENABLE_SPEC_AUGMENT else 0.0
    model.config.mask_feature_prob  = 0.0
    model.freeze_feature_encoder()  # CNN feature extractor frozen, transformer fine-tuned
    return model, fe


def build_optimizer_scheduler(model, n_train):
    head_ids   = {id(p) for p in model.lm_head.parameters()}
    enc_params = [p for p in model.parameters()
                  if id(p) not in head_ids and p.requires_grad]
    hd_params  = [p for p in model.lm_head.parameters() if p.requires_grad]
    total_steps = math.ceil(n_train / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
    warmup      = int(total_steps * WARMUP_RATIO)
    opt = torch.optim.AdamW(
        [{'params': enc_params, 'lr': ENCODER_LR},
         {'params': hd_params,  'lr': HEAD_LR}],
        weight_decay=WEIGHT_DECAY,
    )
    sch = get_linear_schedule_with_warmup(opt, warmup, total_steps)
    print(f'Steps={total_steps}  Warmup={warmup}  '
          f'enc_lr={ENCODER_LR}  head_lr={HEAD_LR}')
    print(f'Enc params: {sum(p.numel() for p in enc_params):,}')
    print(f'Head params: {sum(p.numel() for p in hd_params):,}')
    return opt, sch

model, fe = build_model()
model.to(DEVICE)
print(f'Model loaded: {BASE_MODEL}')

In [ ]:
# ── Inference helpers (dùng chung cho validation callback + test) ────────────
def _load_audio(path):
    y, sr = load_wav_f32(str(path))
    assert sr == 16000, f'Expected 16kHz, got {sr}: {path}'
    return trim_silence(y, 16000)

def _greedy_ctc_with_conf(logits):
    all_preds, all_confs = [], []
    for seq in logits:
        probs    = scipy.special.softmax(seq, axis=-1)
        pred_ids = np.argmax(seq, axis=-1)
        out_ph, out_cf, prev = [], [], None
        for t, i in enumerate(pred_ids):
            if i == prev: continue
            prev = int(i)
            if prev == BLANK_ID: continue
            out_ph.append(id2phone[prev] if prev < len(id2phone) else UNK)
            out_cf.append(float(probs[t, prev]))
        all_preds.append(' '.join(out_ph))
        all_confs.append(out_cf)
    return all_preds, all_confs

def run_batched(mdl, feat_e, paths, tag=''):
    all_preds, all_confs = [], []
    n = len(paths)
    mdl.eval()
    for start in range(0, n, INFER_BATCH):
        batch  = [_load_audio(p) for p in paths[start:start+INFER_BATCH]]
        inputs = feat_e(batch, sampling_rate=16000, return_tensors='pt', padding=True)
        iv   = inputs.input_values.to(DEVICE)
        attn = inputs.get('attention_mask')
        if attn is not None: attn = attn.to(DEVICE)
        with torch.no_grad():
            logits = mdl(iv, attention_mask=attn).logits.cpu().numpy()
        p, c = _greedy_ctc_with_conf(logits)
        all_preds.extend(p)
        all_confs.extend(c)
        print(f'  {tag}: {min(start+INFER_BATCH, n)}/{n}', end='\r', flush=True)
    print()
    return all_preds, all_confs

print('Inference helpers ready.')

In [ ]:
# ── EpochMetricsCallback ─────────────────────────────────────────────────────
class EpochMetricsCallback(TrainerCallback):
    def __init__(self, val_paths, gt_c, gt_t, feat_ext):
        self._val_paths  = val_paths
        self._gt_c       = gt_c
        self._gt_t       = gt_t
        self._fe         = feat_ext
        self._best_score = -1.0
        self._best_epoch = 0
        self.history     = []

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        preds, _ = run_batched(model, self._fe, self._val_paths, tag='Valid')
        m = evaluate_on_valid(self._gt_c, self._gt_t, preds,
                              tag=f'Epoch {int(state.epoch):2d}')
        m['epoch'] = int(state.epoch)
        self.history.append(m)

        if m['score'] > self._best_score:
            self._best_score = m['score']
            self._best_epoch = int(state.epoch)
            model.save_pretrained(str(BEST_CKP_DIR))
            self._fe.save_pretrained(str(BEST_CKP_DIR))
            print(f'  -> Best  epoch={self._best_epoch}  score={self._best_score:.4f}')

        model.train()

print('Callback ready.')

In [ ]:
# ── Training ─────────────────────────────────────────────────────────────────
train_ds = MDDDataset(train_df, train_ph, is_train=True)
valid_ds = MDDDataset(valid_df, valid_ph, is_train=False)
collator = MDDCollator(fe=fe)
opt, sch = build_optimizer_scheduler(model, len(train_df))
cb       = EpochMetricsCallback(val_paths, VALID_C, VALID_T, fe)

training_args = TrainingArguments(
    output_dir                  = str(OUT_DIR / 'checkpoints'),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    fp16                        = torch.cuda.is_available(),
    eval_strategy               = 'no',
    save_strategy               = 'no',
    group_by_length             = True,
    remove_unused_columns       = False,
    report_to                   = 'none',
    logging_steps               = 30,
    seed                        = SEED,
    label_names                 = ['labels'],
)
trainer = MDDTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_ds,
    eval_dataset  = valid_ds,
    data_collator = collator,
    callbacks     = [cb],
    optimizers    = (opt, sch),
)

print(f'Training {EPOCHS} epochs | base: {BASE_MODEL}')
print(f'Train: {len(train_ds)}  Valid: {len(valid_ds)}')
trainer.train()

# Save history
with open(OUT_DIR / 'history.json', 'w') as f:
    json.dump(cb.history, f, indent=2)

print(f'\nBest: epoch={cb._best_epoch}  score={cb._best_score:.4f}')
print(f'Best checkpoint: {BEST_CKP_DIR}')

In [ ]:
# ── Training history ─────────────────────────────────────────────────────────
hist_df = pd.DataFrame(cb.history)[['epoch','f1','per','der','score']]
print(hist_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

In [ ]:
# ── Load best checkpoint ─────────────────────────────────────────────────────
print(f'Loading best checkpoint (epoch {cb._best_epoch}) ...')
best_model = Wav2Vec2ForCTC.from_pretrained(str(BEST_CKP_DIR)).to(DEVICE).eval()
best_fe    = AutoFeatureExtractor.from_pretrained(str(BEST_CKP_DIR))
print('Done.')

In [ ]:
# ── FP rate calibration (validation set) ────────────────────────────────────
print('── Computing per-phoneme FP rates on validation set ──')
val_preds, _ = run_batched(best_model, best_fe, val_paths, 'Valid')

occ, fp_count = Counter(), Counter()
for c in VALID_C:
    for tok in c.replace('*','').replace('$','').split():
        occ[tok] += 1

for c, t, p in zip(VALID_C, VALID_T, val_preds):
    rs, hs, op_rh   = _align_pair(c, t)
    hs2, os2, op_ho = _align_pair(t, p)
    flag = 0
    for i in range(len(hs)):
        if hs[i] == '<eps>': continue
        while flag < len(hs2) and hs2[flag] == '<eps>': flag += 1
        if flag < len(hs2) and hs[i] == hs2[flag]:
            if op_rh[i] == 'C' and op_ho[flag] != 'C':
                key = rs[i] if rs[i] != '<eps>' else '<blank_pos>'
                fp_count[key] += 1
            flag += 1

fp_rates  = {ph: fp_count[ph]/occ[ph] for ph in fp_count if occ.get(ph,0)>0}
eligible  = [(ph,r) for ph,r in fp_rates.items() if fp_count.get(ph,0) >= MIN_FP]
sorted_ph = [ph for ph,_ in sorted(eligible, key=lambda x: -x[1])]
suppress  = set(sorted_ph[:K_SUPPRESS])

print(f'Eligible: {len(eligible)} phonemes | K={K_SUPPRESS} suppressed')
print(f'Top-10 suppressed: {sorted_ph[:10]}')

In [ ]:
# ── Public test inference + K=50 calibration ─────────────────────────────────
def calibrate(gt_c, preds, confs, suppress_set):
    out = []
    for c, pred, conf in zip(gt_c, preds, confs):
        if not pred.strip():
            out.append(c); continue
        ca, pa = _align_pair(c, pred)[:2]
        conf_map = dict(enumerate(conf))
        new_tokens, p_idx = [], 0
        for ct, pt in zip(ca, pa):
            if pt == '<eps>': continue
            cur_conf = conf_map.get(p_idx, 1.0)
            p_idx += 1
            if ct != '<eps>' and pt != ct:
                thr = 1.0 if ct in suppress_set else DEF_THR
                new_tokens.append(ct if cur_conf < thr else pt)
            else:
                new_tokens.append(pt)
        out.append(' '.join(new_tokens))
    return out

print('── Public test inference ──')
test_df    = pd.read_csv(PUBTEST_META / 'public_test_phones.csv')
test_paths = [PUBTEST_AUDIO / Path(r['path']).name for _, r in test_df.iterrows()]
test_canon = test_df['canonical'].tolist()
test_trans = test_df['transcript'].tolist()

missing = [p for p in test_paths if not Path(p).exists()]
if missing:
    print(f'WARNING: {len(missing)} files not found')

test_preds, test_confs = run_batched(best_model, best_fe, test_paths, 'Test')
calibrated = calibrate(test_canon, test_preds, test_confs, suppress)
print('Calibration done.')

In [ ]:
# ── Score on public test ─────────────────────────────────────────────────────
m_cal  = evaluate_on_valid(test_canon, test_trans, calibrated,
                           tag='Public test (K=50 calibrated)')
m_base = evaluate_on_valid(test_canon, test_trans, test_preds,
                           tag='Public test (baseline)       ')

print('\n' + '='*65)
print(f'{"":25s}  {"F1":>7}  {"PER":>7}  {"DER":>7}  {"Score":>7}')
print('-'*65)
print(f'{"Baseline":25s}  {m_base["f1"]:7.4f}  {m_base["per"]:7.4f}  '
      f'{m_base["der"]:7.4f}  {m_base["score"]:7.4f}')
print(f'{"K=50 calibrated":25s}  {m_cal["f1"]:7.4f}  {m_cal["per"]:7.4f}  '
      f'{m_cal["der"]:7.4f}  {m_cal["score"]:7.4f}')
print('='*65)

In [ ]:
# ── Write result_public.csv ──────────────────────────────────────────────────
with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['id', 'path', 'predict'])
    for i, (_, row) in enumerate(test_df.iterrows()):
        w.writerow([row['id'], row['path'], calibrated[i]])

n_diff = sum(1 for c, p in zip(test_canon, calibrated) if c != p)
print(f'Written: {OUTPUT_CSV}')
print(f'Total rows: {len(calibrated)}')
print(f'Flagged as error: {n_diff}/{len(calibrated)} ({n_diff/len(calibrated)*100:.1f}%)')

# Preview
pd.read_csv(OUTPUT_CSV).head(5)